# RAG Pipeline — Build & Deploy

This notebook handles the **setup, compilation, and deployment** of the RAG pipeline on Red Hat OpenShift AI (RHOAI).

## Pipeline Overview

The pipeline has **up to 5 steps**, orchestrated by Kubeflow Pipelines (KFP):

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Parse & Chunk** | RayJob distributes PDF parsing via Docling + HybridChunker, writes JSONL to S3 (MinIO) |
| 2 | **Deploy Embedding** *(optional)* | Deploys embedding model as a KServe InferenceService (only when `deploy_embedding=True`) |
| 3 | **Ingest to Milvus** | Reads chunks from S3, embeds with local or deployed model, inserts into Milvus |
| 4 | **Download Model** | Downloads LLM weights to a PVC (cached — skips if already present) |
| 5 | **Deploy Model** | Creates a vLLM InferenceService from the cached PVC model |

Steps 1→3 form the **data chain** and steps 4→5 form the **model chain** — both chains run in parallel.

## Embedding Modes

This pipeline supports **two modes** for generating embeddings:

**Mode 1: Local** (default)
- Embedding model runs inside the pipeline component
- Uses `sentence-transformers` library
- Simpler setup, good for testing

**Mode 2: Service**
- Embedding model deployed as a dedicated InferenceService
- Reusable across multiple runs and queries
- Better resource utilization, can leverage GPU
- Ideal for production and If you're planning to use [OGX](https://ogx-ai.github.io/)

Toggle between modes by setting `EMBEDDING_MODE = "local"` or `"service"` in the configuration.

## What This Notebook Covers

1. **Prerequisites** — Create PVCs, Secrets, RBAC
2. **Compile the Pipeline** — Generate the KFP YAML
3. **Upload & Run** — Submit to Data Science Pipelines
4. **Monitor** — Track pipeline execution
5. **Cleanup** — Tear down resources

---
## 1. Prerequisites

Before running the pipeline, ensure the following are in place:

- **RHOAI** installed with Data Science Pipelines (DSPA) configured
- **KubeRay** operator enabled
- **Milvus** deployed (e.g., via Milvus Operator)
- **MinIO** or S3-compatible storage for intermediate chunks
- **GPU nodes** with NVIDIA GPUs for model serving

### Authentication

Provide your OpenShift API server URL and OAuth token.
Get your token with: `oc whoami -t`

In [ ]:
# Install required packages
!pip install -q kfp kfp-kubernetes kubernetes codeflare-sdk tabulate requests

In [ ]:
from rag_setup import RAGSetup

API_SERVER = "https://api.your-cluster.example.com:6443"  # oc whoami --show-server
TOKEN = "sha256~xxxxxxxx"  # oc whoami -t
NAMESPACE = "ray-docling"

setup = RAGSetup(api_server=API_SERVER, token=TOKEN, namespace=NAMESPACE)

In [ ]:
# ============================================================
# Configuration — Update these values for your environment
# ============================================================

# S3 / MinIO
S3_ENDPOINT = "http://minio-service.default.svc.cluster.local:9000"
S3_BUCKET = "open-rag-bench-chunks"
S3_PREFIX = "chunks"
S3_SECRET_NAME = "minio-secret"  # K8s Secret with keys: access_key, secret_key

# PDF Input
INPUT_PATH = "open_rag_pdfs"  # Path relative to pvc_mount_path

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "open_rag_pdfs"
DROP_EXISTING = True  # If True, drop and recreate Milvus collection on each run

# Embedding Configuration
EMBEDDING_MODE = "local"  # Options: "local" or "service"
EMBEDDING_MODEL = "ibm-granite/granite-embedding-125m-english"
EMBEDDING_DIM = 768
# Only used when EMBEDDING_MODE = "service"
EMBEDDING_SERVICE_NAME = "embedding-service"
EMBEDDING_ENDPOINT = (
    f"http://{EMBEDDING_SERVICE_NAME}.{NAMESPACE}.svc.cluster.local:8080"
)

# LLM
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# InferenceService name (derived from model name, must match model_deployment component logic)
LLM_ISVC_NAME = "mistral-7b-instruct-v0-3"
MODEL_CACHE_PVC = "model-cache-pvc"
HF_SECRET_NAME = "hf-token-secret"  # K8s Secret with key: token (HuggingFace API token)
# KServe RawDeployment auto-creates a predictor service: {isvc-name}-predictor
INFERENCE_URL = (
    f"http://{LLM_ISVC_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8080/v1"
)

# Ray / Kueue
BYPASS_KUEUE = (
    False  # Set True on clusters without Kueue or with sufficient free resources
)

print(f"Namespace:      {NAMESPACE}")
print(f"S3 endpoint:    {S3_ENDPOINT}")
print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Drop existing:  {DROP_EXISTING}")
print(f"Input path:     {INPUT_PATH}")
print(f"Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"Embedding URL:  {EMBEDDING_ENDPOINT}")
else:
    print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM ISVC:       {LLM_ISVC_NAME}")
print(f"LLM endpoint:   {INFERENCE_URL}")
print(f"HF secret:      {HF_SECRET_NAME}")
print(f"Bypass Kueue:   {BYPASS_KUEUE}")

### 1.1 Create S3 Secret

In [ ]:
setup.create_secret(
    S3_SECRET_NAME,
    {
        "access_key": "minioadmin",
        "secret_key": "minioadmin",
    },
)

### 1.2 Create HuggingFace Token Secret (Optional)

If your LLM requires authentication (e.g., gated models like Mistral, Llama), create a Secret with your HuggingFace token. The pipeline injects this token into the model-download pod — **if the Secret doesn't exist, the pod won't be able to pull gated model weights**.

> **Skip this step** if you're using a public (non-gated) model.

You can also create this secret from the command line:
```bash
oc create secret generic hf-token-secret \
  --from-literal=token=<your-huggingface-token> \
  -n ray-docling
```

In [ ]:
# Replace <your-huggingface-token> with your actual token
# Get one at https://huggingface.co/settings/tokens
setup.create_secret(
    HF_SECRET_NAME,
    {"token": "<your-huggingface-token>"},
)

### 1.3 Create Model Cache PVC

In [ ]:
setup.create_pvc(MODEL_CACHE_PVC, size="30Gi")

### 1.4 Create RBAC

In [ ]:
setup.setup_pipeline_rbac()

### 1.5 Verify Prerequisites

In [ ]:
setup.verify_prerequisites(
    secret_name=S3_SECRET_NAME,
    pvc_name=MODEL_CACHE_PVC,
    hf_secret_name=HF_SECRET_NAME,
)

---
## 2. Compile the Pipeline

The pipeline is defined in [pipelines-components/pipelines/data_processing/ray_data
/pdf_documents_processing_rag_pipeline](https://github.com/opendatahub-io/pipelines-components/blob/main/pipelines/data_processing/ray_data/pdf_documents_processing_rag_pipeline/pipeline.py). Compile it to generate the KFP YAML.

First, let's install pipeline components to compile the pipeline.

In [ ]:
!pip install git+https://github.com/opendatahub-io/pipelines-components.git

In [ ]:
from kfp import compiler
from kfp_components.pipelines.data_processing.ray_data.pdf_documents_processing_rag_pipeline.pipeline import (
    rag_multistep_pipeline,
)

compiler.Compiler().compile(
    rag_multistep_pipeline,
    package_path="rag_multistep_pipeline.yaml",
)
print("Pipeline compiled to rag_multistep_pipeline.yaml")

---
## 3. Upload & Run the Pipeline

In [ ]:
kfp_client = setup.get_kfp_client()

pipeline = kfp_client.upload_pipeline(
    pipeline_package_path="rag_multistep_pipeline.yaml",
    pipeline_name="rag-multi-step-pipeline",
)
print(f"Pipeline uploaded: {pipeline.pipeline_id}")

### Create Pipeline Run

In [ ]:
# Determine embedding endpoint and deploy flag
embedding_endpoint_param = EMBEDDING_ENDPOINT if EMBEDDING_MODE == "service" else ""
deploy_embedding_param = EMBEDDING_MODE == "service"

# Create run
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file="rag_multistep_pipeline.yaml",
    run_name="rag-test-run",
    arguments={
        "pvc_name": "data-pvc",
        "pvc_mount_path": "/mnt/data",
        "namespace": NAMESPACE,
        "s3_endpoint": S3_ENDPOINT,
        "s3_bucket": S3_BUCKET,
        "s3_prefix": S3_PREFIX,
        "s3_secret_name": S3_SECRET_NAME,
        "input_path": INPUT_PATH,
        "ray_image": "quay.io/rhoai-szaher/docling-ray:latest",
        "num_workers": 2,
        "worker_cpus": 8,
        "worker_memory_gb": 16,
        "cpus_per_actor": 4,
        "min_actors": 2,
        "max_actors": 4,
        "chunk_max_tokens": 256,
        "num_files": 1000,
        "bypass_kueue": BYPASS_KUEUE,
        "deploy_embedding": deploy_embedding_param,
        "embedding_endpoint": embedding_endpoint_param,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dim": EMBEDDING_DIM,
        "milvus_host": MILVUS_HOST,
        "milvus_port": MILVUS_PORT,
        "collection_name": COLLECTION_NAME,
        "drop_existing": DROP_EXISTING,
        "llm_model_name": LLM_MODEL_NAME,
        "hf_secret_name": HF_SECRET_NAME,
        "model_cache_pvc": MODEL_CACHE_PVC,
        "max_model_len": 4096,
        "gpu_count": 1,
    },
)

print(f"Run started: {run.run_id}")
print("\nConfiguration:")
print(f"  PDFs from: /mnt/data/{INPUT_PATH}")
print(f"  Embedding mode: {EMBEDDING_MODE}")
print(f"  Deploy embedding: {deploy_embedding_param}")
if EMBEDDING_MODE == "service":
    print(f"  Embedding endpoint: {embedding_endpoint_param}")
else:
    print(f"  Embedding model: {EMBEDDING_MODEL} (local)")
print(f"  Drop existing collection: {DROP_EXISTING}")
print(f"  Bypass Kueue: {BYPASS_KUEUE}")

---
## 4. Monitor Pipeline Execution

### Step 1: Parse & Chunk (RayJob)

In [ ]:
setup.show_rayjobs()

### Step 2: Pipeline Pods

In [ ]:
setup.show_pipeline_pods()

### Steps 3 & 4: Model Download & Deployment

In [ ]:
setup.show_inferenceservices(llm_isvc_name=LLM_ISVC_NAME)

---
## Next Steps

Once the pipeline completes successfully, use the [rag_query_test.ipynb](../test/rag_query_test.ipynb) notebook to:
- Verify the deployed services
- Test RAG queries
- Explore the RAG system interactively

You can also try [rag_ogx_streaming.ipynb](../ogx_test/rag_ogx_streaming.ipynb) for streaming responses using OpenGenAI Stack (OGX).

---
## 5. Cleanup (Optional)

Uncomment the sections below to tear down deployed resources.

In [ ]:
# Uncomment to tear down all RAG resources:

# setup.cleanup_all(
#     llm_isvc_name=LLM_ISVC_NAME,
#     embedding_service_name=EMBEDDING_SERVICE_NAME,
#     pvc_name=MODEL_CACHE_PVC,
#     secret_name=S3_SECRET_NAME,
# )

# Or clean up individually:
# setup.delete_inferenceservice(LLM_ISVC_NAME)
# setup.delete_servingruntime(LLM_ISVC_NAME)
# setup.delete_inferenceservice(EMBEDDING_SERVICE_NAME)
# setup.delete_servingruntime(EMBEDDING_SERVICE_NAME)
# setup.delete_pvc(MODEL_CACHE_PVC)
# setup.delete_secret(S3_SECRET_NAME)
# setup.delete_rbac()

print("Uncomment the sections above to execute cleanup")